# NST_Proj V3 - PyTorch Transformer (SegFormer/Swin) Training on Colab

Notebook này huấn luyện mô hình Teacher-Student (Mean Teacher) bằng **PyTorch** và **HuggingFace Transformers**.
Bao gồm 2 thí nghiệm để làm Ablation Study: **Standard Loss (Baseline)** và **Fuzzy Loss (Proposed Method)**.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cài đặt các thư viện cần thiết
!pip install -q transformers opencv-python pandas matplotlib torch torchvision

In [ ]:
# Tải dự án từ GitHub
!rm -rf /content/nst_proj_v2
!git clone --depth 1 https://github.com/Killua-2002/nst_proj_2.git /content/nst_proj_v2

In [ ]:
# Giải nén Dataset
%cd /content/nst_proj_v2
!mkdir -p data/processed
!unzip -q dataset.zip -d data/processed/

---
## 🧪 THÍ NGHIỆM 1: MÔ HÌNH BASELINE (Standard BCE + Dice Loss)
Đây là mô hình Teacher-Student dùng hàm Loss truyền thống để làm mốc so sánh.

In [ ]:
%cd /content/nst_proj_v2
# Chạy huấn luyện Baseline (Lưu kết quả về Google Drive)
!python scripts/train_baseline.py \
    --dataset-dir data/processed \
    --results-dir /content/drive/MyDrive/nst_experiments_baseline_pytorch \
    --epochs 200 \
    --patience 25 \
    --batch-size 128 \
    --lr 1e-4 \
    --model-type segformer

In [ ]:
# Vẽ biểu đồ cho Baseline
!python scripts/plot_curves.py --csv /content/drive/MyDrive/nst_experiments_baseline_pytorch/ts_standard_segformer/epoch_log.csv --out /content/drive/MyDrive/nst_experiments_baseline_pytorch/ts_standard_segformer/learning_curve_baseline.png

from IPython.display import Image
Image('/content/drive/MyDrive/nst_experiments_baseline_pytorch/ts_standard_segformer/learning_curve_baseline.png')

---
## 🚀 THÍ NGHIỆM 2: MÔ HÌNH ĐỀ XUẤT (Fuzzy Loss)
Đây là mô hình Teacher-Student dùng hàm Fuzzy Logic Loss (Phương pháp mới của bạn) để so sánh với Baseline ở trên.

In [ ]:
%cd /content/nst_proj_v2
# Chạy huấn luyện Fuzzy (Lưu kết quả về Google Drive)
!python scripts/train_fuzzy_ts.py \
    --dataset-dir data/processed \
    --results-dir /content/drive/MyDrive/nst_experiments_fuzzy_pytorch \
    --epochs 200 \
    --patience 25 \
    --batch-size 128 \
    --lr 1e-4 \
    --model-type segformer

In [ ]:
# Vẽ biểu đồ cho Fuzzy Loss
!python scripts/plot_curves.py --csv /content/drive/MyDrive/nst_experiments_fuzzy_pytorch/ts_FUZZY_segformer/epoch_log.csv --out /content/drive/MyDrive/nst_experiments_fuzzy_pytorch/ts_FUZZY_segformer/learning_curve_fuzzy.png

from IPython.display import Image
Image('/content/drive/MyDrive/nst_experiments_fuzzy_pytorch/ts_FUZZY_segformer/learning_curve_fuzzy.png')

---
## 📊 ĐÁNH GIÁ (EVALUATION)
Chạy dự đoán trên tập Validation và xuất ảnh trực quan.

In [ ]:
%cd /content/nst_proj_v2
# Evaluate on held-out TEST set (last 10% = 700 samples, never seen during training)

# Baseline
!python testing/evaluate.py \
    --weights /content/drive/MyDrive/nst_experiments_baseline_pytorch/ts_standard_segformer/best_model.pth \
    --out-dir /content/drive/MyDrive/nst_experiments_baseline_pytorch/predictions \
    --split test

# Fuzzy
!python testing/evaluate.py \
    --weights /content/drive/MyDrive/nst_experiments_fuzzy_pytorch/ts_FUZZY_segformer/best_model_fuzzy.pth \
    --out-dir /content/drive/MyDrive/nst_experiments_fuzzy_pytorch/predictions \
    --split test
